# Elevator RL Simulation Demo

Run the trained PPO model and visualize it step-by-step in the browser.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch
from sb3_contrib import MaskablePPO
from elevator_env import ElevatorEnv

%matplotlib inline

## Load the trained model and create the environment

In [ ]:
model = MaskablePPO.load('modell/ppo_elevator_episode_1318.zip')

# spawn_intervall=1800 makes guests arrive 4x faster than default (7200)
# so you see meaningful elevator decisions sooner
env = ElevatorEnv(
    render_mode=None,
    num_elevators=3,
    num_floors=10,
    max_passengers=5,
    max_guests=30,
    spawn_intervall=1800,
)
print(f'Environment ready: {env.num_elevators} elevators, {env.num_floors} floors, {env.max_guests} guests')
print(f'Guests arrive ~{env.max_guests / env.spawn_intervall * 3600:.0f}/hour')

## Helper: extract single-elevator observation for the trained model

In [ ]:
def get_single_elevator_obs(full_obs, elevator_idx, num_floors):
    elev_size = 2 + num_floors
    start = elevator_idx * elev_size
    elev_obs = list(full_obs[start:start + elev_size])
    for _ in range(2):
        elev_obs.extend([0] * elev_size)
    waiting_start = len(full_obs) - num_floors
    elev_obs.extend(full_obs[waiting_start:])
    return np.array(elev_obs, dtype=np.int32)

## Visualization function (matplotlib, browser-friendly)

In [ ]:
def render_frame(env, fig, ax):
    ax.clear()
    num_floors = env.num_floors
    num_elevators = env.num_elevators
    
    ax.set_xlim(-1, num_elevators + 2)
    ax.set_ylim(-0.5, num_floors + 0.5)
    ax.set_yticks(range(num_floors))
    ax.set_yticklabels([f'Floor {i}' for i in range(num_floors)])
    ax.set_xticks([])
    
    # Floor lines
    for f in range(num_floors + 1):
        ax.axhline(y=f - 0.5, color='gray', linewidth=0.5, linestyle='--')
    
    # Elevators
    for idx, elev in enumerate(env.elevators):
        color = '#4CAF50' if elev.door_open else '#2196F3'
        rect = FancyBboxPatch(
            (idx + 0.1, elev.current_floor - 0.35), 0.8, 0.7,
            boxstyle='round,pad=0.05', facecolor=color, edgecolor='black'
        )
        ax.add_patch(rect)
        ax.text(idx + 0.5, elev.current_floor, f'{len(elev.passengers)}',
                ha='center', va='center', fontsize=14, fontweight='bold', color='white')
    
    # Waiting guests per floor
    for floor in range(num_floors):
        count = sum(1 for g in env.waiting_guests if g.current_floor == floor)
        if count > 0:
            ax.text(num_elevators + 0.5, floor, f'\u23f3 {count}',
                    ha='center', va='center', fontsize=11, color='red')
    
    # Stats
    sim_seconds = env.episode_steps
    hours = 8 + sim_seconds // 3600
    minutes = (sim_seconds % 3600) // 60
    stats = (
        f'Time: {hours:02}:{minutes:02}  |  '
        f'Waiting: {len(env.waiting_guests)}  |  '
        f'In elevator: {len(env.guests_in_elevator)}  |  '
        f'Left: {env.guests_left_building}/{env.max_guests}  |  '
        f'Reward: {env.total_reward:.0f}'
    )
    ax.set_title(stats, fontsize=10)
    ax.set_xlabel('Elevators (blue=closed, green=open)', fontsize=9)

## Run the simulation with live visualization

Updates every 50 steps. The elevators will idle initially (bouncing between floors) until guests start arriving — that's normal trained behavior. Once guests spawn, you'll see them open doors, board, and deliver.

Interrupt the kernel to stop early.

In [ ]:
obs, info = env.reset()
terminated = False
truncated = False
step_count = 0
render_every = 50

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plt.close(fig)

while not terminated and not truncated:
    action_masks = info['action_mask']
    actions = []
    for i in range(env.num_elevators):
        elev_obs = get_single_elevator_obs(obs, i, env.num_floors)
        action, _ = model.predict(elev_obs, action_masks=action_masks[i], deterministic=True)
        actions.append(int(action[0]))
    
    obs, reward, terminated, truncated, info = env.step(actions)
    step_count += 1
    
    if step_count % render_every == 0:
        clear_output(wait=True)
        render_frame(env, fig, ax)
        display(fig)

# Final frame
clear_output(wait=True)
render_frame(env, fig, ax)
display(fig)
print(f'\nDone! {step_count} steps, {env.guests_left_building}/{env.max_guests} guests processed')

## Results

In [ ]:
import pandas as pd

df = pd.DataFrame(env.logs)
wait_df = df[df['mode'] == 'elevator_waiting']
drive_df = df[df['mode'] == 'elevator_drive']

print(f'Total log entries: {len(df)}')
print(f'Avg wait time:  {wait_df["wait_time"].mean():.1f}s')
print(f'Avg ride time:  {drive_df["travel_time"].mean():.1f}s')
print(f'Avg total time: {(drive_df["wait_time"] + drive_df["travel_time"]).mean():.1f}s')
print(f'Total reward:   {env.total_reward:.0f}')

In [ ]:
if not drive_df.empty:
    fig2, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    drive_df = drive_df.copy()
    drive_df['hour'] = (drive_df['time'] // 3600).astype(int)
    drive_df['total_time'] = drive_df['wait_time'] + drive_df['travel_time']
    
    hourly = drive_df.groupby('hour').agg(
        avg_wait=('wait_time', 'mean'),
        avg_ride=('travel_time', 'mean'),
        avg_total=('total_time', 'mean'),
    ).reset_index()
    
    axes[0].bar(hourly['hour'], hourly['avg_wait'], color='#FF9800')
    axes[0].set_title('Avg Wait Time per Hour')
    axes[0].set_ylabel('Seconds')
    
    axes[1].bar(hourly['hour'], hourly['avg_ride'], color='#2196F3')
    axes[1].set_title('Avg Ride Time per Hour')
    
    axes[2].bar(hourly['hour'], hourly['avg_total'], color='#4CAF50')
    axes[2].set_title('Avg Total Time per Hour')
    
    for a in axes:
        a.set_xlabel('Hour')
    
    plt.tight_layout()
    plt.show()